# Gen AI Reasoning & Recommendation Layer
### Sales Forecasting & Gen AI Project

## 1. Imports

In [2]:
import json
import pandas as pd

In [3]:
from sqlalchemy import create_engine

In [4]:
from google import genai

## 2. API Key Setup

Apni key yahan daalo (ya environment variable `ANTHROPIC_API_KEY` set kar lo aur
`Anthropic()` bina argument ke call karo — wo automatically utha lega).


In [5]:
API_KEY = "enter your api key "

client =genai.Client(api_key=API_KEY)


## 3. Paste Your gap_summary

In [6]:
gap_summary = {
    "target_amount": 400000000,
    "target_year": 2026,
    "actual_so_far": 0,         
    "projected_total": 250633371.55,
    "gap_amount":149366628.45,
    "gap_pct": 37.34,
    "current_growth_rate_pct": 5.85,
    "required_growth_rate_pct": 135.5,
    "top_underperforming_regions":[
    {
      "region": "east",
      "shortfall": -79421932.06374969
    },
    {
      "region": "south",
      "shortfall": -54471705.84881942
    }],
    "top_underperforming_categories": [
    {
      "category": "home & kitchen",
      "shortfall": -102679443.13421425
    },
    {
      "category": "sports",
      "shortfall": -66924789.75354965
    }]
}

print(json.dumps(gap_summary, indent=2))


{
  "target_amount": 400000000,
  "target_year": 2026,
  "actual_so_far": 0,
  "projected_total": 250633371.55,
  "gap_amount": 149366628.45,
  "gap_pct": 37.34,
  "current_growth_rate_pct": 5.85,
  "required_growth_rate_pct": 135.5,
  "top_underperforming_regions": [
    {
      "region": "east",
      "shortfall": -79421932.06374969
    },
    {
      "region": "south",
      "shortfall": -54471705.84881942
    }
  ],
  "top_underperforming_categories": [
    {
      "category": "home & kitchen",
      "shortfall": -102679443.13421425
    },
    {
      "category": "sports",
      "shortfall": -66924789.75354965
    }
  ]
}


## 4. Build the Prompt

In [16]:
def build_prompt(summary):
    return f'''You are a sales performance analyst. Use ONLY the numbers given below —
do not recalculate or invent any numbers, only interpret and explain them.

DATA:
{json.dumps(summary, indent=2)}

Write a short business report with these 4 sections:
1. A 2-3 sentence summary of the gap to target.
2. The top underperforming regions/categories and a plausible business reason.
3. Three concrete, specific actions ranked by expected impact.
4. Which customer segments or areas to prioritize and why
5. Remenber the numbers are in Cr/Lakhs

Keep it concise and business-friendly, no more than 250 words.'''

prompt_text = build_prompt(gap_summary)
print(prompt_text)


You are a sales performance analyst. Use ONLY the numbers given below —
do not recalculate or invent any numbers, only interpret and explain them.

DATA:
{
  "target_amount": 400000000,
  "target_year": 2026,
  "actual_so_far": 0,
  "projected_total": 250633371.55,
  "gap_amount": 149366628.45,
  "gap_pct": 37.34,
  "current_growth_rate_pct": 5.85,
  "required_growth_rate_pct": 135.5,
  "top_underperforming_regions": [
    {
      "region": "east",
      "shortfall": -79421932.06374969
    },
    {
      "region": "south",
      "shortfall": -54471705.84881942
    }
  ],
  "top_underperforming_categories": [
    {
      "category": "home & kitchen",
      "shortfall": -102679443.13421425
    },
    {
      "category": "sports",
      "shortfall": -66924789.75354965
    }
  ]
}

Write a short business report with these 4 sections:
1. A 2-3 sentence summary of the gap to target.
2. The top underperforming regions/categories and a plausible business reason.
3. Three concrete, specific act

## 5. Call the API

In [9]:
from google.genai import types

In [17]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_text,
    config=types.GenerateContentConfig(
        max_output_tokens=2000
    )
)

ai_output = response.text
print(ai_output)

Here is a short business report based on the provided data:

**Sales Performance Report**

**1. Summary of the Gap to Target**
The company is targeting a sales amount of 400 Cr by 2026. Current projections indicate a total of 250.63 Cr, resulting in a significant shortfall. This leaves a gap of 149.37 Cr, representing 37.34% of the overall target.

**2. Top Underperforming Regions/Categories and a Plausible Business Reason**
The East region shows the largest shortfall at -79.42 Cr, followed by the South region at -54.47 Cr. Concurrently, "Home & Kitchen" is the most underperforming category with a -102.68 Cr shortfall, ahead of "Sports" at -66.92 Cr. A plausible business reason for these regional and category shortfalls could be increased market competition or ineffective localized marketing efforts.

**3. Three Concrete, Specific Actions Ranked by Expected Impact**
1.  **Highest Impact:** Develop and launch targeted sales and marketing campaigns specifically for the "Home & Kitchen" c